In [ ]:
import pandas as pd
import numpy as np
from scipy.io import loadmat
import statsmodels.formula.api as smf
from sklearn.svm import SVR
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import mean_squared_error, r2_score
from nilearn.decoding import Decoder

# Path to the 120-row Likert data file (used for Y label calculation)
pheno_file_path = '../data/Likert_scales_fMRI_Romy.xlsx' 
# Path to the PG matrix file (X input for decoding)
pg_file_path = '../data/all_embedding.mat'

# Load Pheno Data 
try:
    Likert_df = pd.read_excel(pheno_file_path)
    print(f"Phenomenal data loaded successfully. Rows: {len(Likert_df)}")
except FileNotFoundError:
    print(f"ERROR: Pheno data not found at {pheno_file_path}")
    Likert_df = None

Phenomenal data loaded successfully. Rows: 120


# Pheno Data Preparation ($\mathbf{Y}$ Vector Creation)

In [21]:
if Likert_df is not None:
    # Compute Aggregated Likert Categories (Raw Scores)
    
    raw_answer_cols = [f'Stateq_Answ{i}' for i in range(1, 18)] 
    for col in raw_answer_cols:
        if col in Likert_df.columns:
            Likert_df[col] = pd.to_numeric(Likert_df[col], errors='coerce').astype('float64')

    Likert_df['difficulty_raw'] = Likert_df['Stateq_Answ1'] + Likert_df['Stateq_Answ2'] + Likert_df['Stateq_Answ3']
    Likert_df['emotion_raw'] = Likert_df['Stateq_Answ4'] + Likert_df['Stateq_Answ5'] + Likert_df['Stateq_Answ12']
    Likert_df['absorp_raw'] = Likert_df['Stateq_Answ11'] + Likert_df['Stateq_Answ13'] + Likert_df['Stateq_Answ14']
    Likert_df['meta_raw'] = Likert_df['Stateq_Answ16'] + Likert_df['Stateq_Answ17']
    Likert_df['realness_raw'] = Likert_df['Stateq_Answ6'] + Likert_df['Stateq_Answ7'] + Likert_df['Stateq_Answ8'] + Likert_df['Stateq_Answ9'] + Likert_df['Stateq_Answ10'] + Likert_df['Stateq_Answ15']


    # Reshape and Calculate Delta Scores
    
    long_data = []
    #Iterate using the BIDS ID column (`Sub_id_fMRI`) for accurate matching
    #the unique list of subjects is defined by the BIDS ID column
    subjects_to_iterate = Likert_df['Sub_id_fMRI'].unique() 

    for subject_bids_id in subjects_to_iterate:
        # Filter the original DataFrame by the BIDS ID
        subj_data = Likert_df[Likert_df['Sub_id_fMRI'] == subject_bids_id].copy()
        
        # Outlier Exclusion: Filter for N=36 clean subjects 
        if 'allcov_ex' in subj_data.columns and subj_data['allcov_ex'].iloc[0] != 1:
            continue 

        # Find Control row (Bloc 1)
        if subj_data[subj_data['Bloc'] == 1].empty: continue
        ctrl_row = subj_data[subj_data['Bloc'] == 1].iloc[0]

        # Process Med and Hyp states (Bloc 2 and 3)
        for index, state_row in subj_data[subj_data['Bloc'] != 1].iterrows():
            
            # Calculate Delta Scores
            d_real = state_row['realness_raw'] - ctrl_row['realness_raw']
            d_absorp = state_row['absorp_raw'] - ctrl_row['absorp_raw'] 
            d_emotion = state_row['emotion_raw'] - ctrl_row['emotion_raw']
            d_meta = state_row['meta_raw'] - ctrl_row['meta_raw']
            d_difficulty = state_row['difficulty_raw'] - ctrl_row['difficulty_raw'] 
            
            # Combined Score: (Realness + Absorp + Emotion) - Meta
            d_combined = d_real + d_absorp + d_emotion - d_meta

            long_data.append({
                'sub_name': subject_bids_id,
                'group': state_row['Version'],
                'state': state_row['state'],
                'combined_pheno_delta': d_combined,
                'difficulty_delta': d_difficulty,
                'run_type': state_row['state'] 
            })

    data_for_analysis = pd.DataFrame(long_data).dropna()
    data_for_analysis.rename(columns={'Version': 'group'}, inplace=True)
    
    # Sort the Y labels to match the PG matrix order (Sub_name then State: H, M)
    Y_labels_data = data_for_analysis.sort_values(by=['sub_name', 'state'], ascending=[True, False]).reset_index(drop=True)

    print(f"Final Phenomenal Data (Y) created. Rows: {len(Y_labels_data)}")

else:
    Y_labels_data = None

Final Phenomenal Data (Y) created. Rows: 80


# PG Data Preparation ($\mathbf{X}$ Input Creation)

In [22]:
if Y_labels_data is not None:
    print("--- Phenomenal Data Subject ID Columns ---")
    
    print("Column 'sub_name' values:")
    print(Y_labels_data['sub_name'].head())

    if 'Sujet' in Y_labels_data.columns:
        print("\nColumn 'Sujet' values:")
        print(Y_labels_data['Sujet'].head())

    if 'Sub_id_fMRI' in Y_labels_data.columns:
        print("\nColumn 'Sub_id_fMRI' values:")
        print(Y_labels_data['Sub_id_fMRI'].head())
    
    if 'sub_id' in Y_labels_data.columns:
        print("\nColumn 'sub_id' values:")
        print(Y_labels_data['sub_id'].head())

--- Phenomenal Data Subject ID Columns ---
Column 'sub_name' values:
0    sub-01
1    sub-01
2    sub-02
3    sub-02
4    sub-03
Name: sub_name, dtype: object


In [23]:
from scipy.io import loadmat

if Y_labels_data is not None:
    
    # Load PG matrix and subject list
    try:
        pg_data = loadmat(pg_file_path)
        all_embs_full = pg_data['emb']  # Shape: (117, 18715, 5)
        
        # extract and flatten subject IDs (strings) from the MATLAB array
        subject_list_full = pg_data['subs'].flatten()
        subject_list_full = [sub[0] if isinstance(sub, np.ndarray) else sub for sub in subject_list_full]

    except Exception as e:
        print(f"ERROR loading PG data: {e}")
        X_data_for_decoding = None
        Y_labels_data = None
        
    if Y_labels_data is not None:
        
        # ----------------------------------------------------------------------
        # ALIGN SUBJECT LISTS (INTERSECTION)
        # ----------------------------------------------------------------------
        
        # Subjects present in the PG matrix
        subjects_in_X = set(subject_list_full)
        
        # Subjects present in the final Phenomenal data (Y)
        subjects_in_Y = set(Y_labels_data['sub_name'].unique())
        
        # The final set of subjects to analyse is the intersection (only subjects present in both)
        final_aligned_subjects = sorted(list(subjects_in_X.intersection(subjects_in_Y)))

        if len(final_aligned_subjects) * 2 != len(Y_labels_data):
            print(f"\nWarning: N of subjects has changed. Old N={len(subjects_in_Y)}, New N={len(final_aligned_subjects)}")

        # ----------------------------------------------------------------------
        # FILTER Y DATA (Pheno)
        # ----------------------------------------------------------------------
        
        # Filter the phenomenal data (Y) to keep only the subjects present in the PG data
        Y_labels_data_filtered = Y_labels_data[Y_labels_data['sub_name'].isin(final_aligned_subjects)].copy()
        
        # Sort Y to guarantee perfect alignment with the X matrix extraction order
        Y_labels_data_filtered = Y_labels_data_filtered.sort_values(by=['sub_name', 'state'], ascending=[True, False]).reset_index(drop=True)

        # ----------------------------------------------------------------------
        # FILTER X DATA (PG)
        # ----------------------------------------------------------------------

        X_pg_list = []
        
        # Iterate through the PG matrix's subjects and extract the rows corresponding to final list
        for i, sub_id in enumerate(subject_list_full):
            if sub_id in final_aligned_subjects:
                # PG matrix observations are: (Control, Meditation, Hypnosis) per subject.
                # We need M (index 1) and H (index 2).
                
                # Extract Meditation (M) - PG (index 0)
                X_pg_list.append(all_embs_full[3 * i + 1, :, 0])
                
                # Extract Hypnosis (H) - PG (index 0)
                X_pg_list.append(all_embs_full[3 * i + 2, :, 0])

        X_data_for_decoding = np.vstack(X_pg_list)
        
        # ----------------------------------------------------------------------
        # FINAL SANITY CHECK
        # ----------------------------------------------------------------------

        # Store the final aligned data for the next cell
        X_data_for_decoding = X_data_for_decoding
        Y_labels_data = Y_labels_data_filtered
        
        print(f"\nAlignment Success! Final N={len(final_aligned_subjects)} subjects ({len(Y_labels_data)} observations).")
        print(f"Final Input Matrix X (PG) Shape: {X_data_for_decoding.shape}")
        
        if X_data_for_decoding.shape[0] == Y_labels_data.shape[0]:
            print("Data dimensions are now perfectly matched. Proceed to decoding in Cell 4.")
        else:
            print("CRITICAL ERROR: Alignment failed despite filtering. Check subject sorting keys.")



Alignment Success! Final N=39 subjects (78 observations).
Final Input Matrix X (PG) Shape: (78, 18715)
Data dimensions are now perfectly matched. Proceed to decoding in Cell 4.


# Final SVC Decoding ($\mathbf{X} \to \mathbf{Y}$)

In [24]:
if X_data_for_decoding is not None:
    
    from sklearn.model_selection import cross_val_score
    from sklearn.svm import SVC, SVR
    
    # Prepare Y labels and CV objects
    Y_combined = Y_labels_data['combined_pheno_delta'].values
    Y_difficulty = Y_labels_data['difficulty_delta'].values
    Y_classification = Y_labels_data['state'].values
    
    # Use LeaveOneGroupOut for cross-validation by Subject (groups are the subject IDs)
    groups = Y_labels_data['sub_name'].values
    cv = LeaveOneGroupOut()
    
    print("\n--- Starting Scikit-learn CV Decoding (PG -> Pheno Scores) ---")
    
    results = {}

    # -----------------------------------------------------------
    # DECODE 1: CLASSIFY STATE TYPE (M vs H) ---
    # Goal: Use PG to classify the state type (M or H)
    # -----------------------------------------------------------
    
    # Estimator: Support Vector Classifier (SVC)
    svc_estimator = SVC(kernel='linear', C=1.0, random_state=42)
    
    # Use cross_val_score to get accuracy scores across all CV folds
    scores_state = cross_val_score(
        svc_estimator, 
        X_data_for_decoding, 
        Y_classification, 
        cv=cv, 
        groups=groups, 
        scoring='accuracy',
        n_jobs=-1
    )
    score_state = scores_state.mean()
    results['Classification_State_M_H'] = f"{score_state:.3f} (Accuracy)"


    # -----------------------------------------------------------
    # DECODE 2: PREDICT COMBINED PHENO SCORE (Regression) ---
    # Goal: Use PG to predict the general absorption/meta-awareness delta
    # -----------------------------------------------------------
    
    # Estimator: Support Vector Regressor (SVR)
    svr_estimator = SVR(kernel='linear', C=1.0)
    
    scores_combined = cross_val_score(
        svr_estimator, 
        X_data_for_decoding, 
        Y_combined, 
        cv=cv, 
        groups=groups, 
        scoring='r2', # R2 is Coefficient of Determination (predictive fit)
        n_jobs=-1
    )
    r2_combined = scores_combined.mean()
    results['Regression_Combined_Score'] = f"{r2_combined:.3f} (R2)"
    
    
    # -----------------------------------------------------------
    # --- DECODE 3: PREDICT DIFFICULTY DELTA (Regression) ---
    # Goal: Use PG to predict the subjective difficulty/order effect
    # -----------------------------------------------------------

    scores_difficulty = cross_val_score(
        svr_estimator, 
        X_data_for_decoding, 
        Y_difficulty, 
        cv=cv, 
        groups=groups, 
        scoring='r2', 
        n_jobs=-1
    )
    r2_difficulty = scores_difficulty.mean()
    results['Regression_Difficulty_Delta'] = f"{r2_difficulty:.3f} (R2)"
    
    
    # --- FINAL REPORT ---
    print("\n=============================================")
    print("           DECODING RESULTS (Phase 2)        ")
    print("=============================================")
    for k, v in results.items():
        print(f"| {k:30} | {v:15} |")
    print("=============================================")
    print("\nResults interpreted as: Higher Accuracy/R2 means PG is more predictive.")
    
else:
    print("Decoding skipped due to data mismatch/loading errors.")


--- Starting Scikit-learn CV Decoding (PG -> Pheno Scores) ---

           DECODING RESULTS (Phase 2)        
| Classification_State_M_H       | 0.526 (Accuracy) |
| Regression_Combined_Score      | -64.079 (R2)    |
| Regression_Difficulty_Delta    | -21.363 (R2)    |

Results interpreted as: Higher Accuracy/R2 means PG is more predictive.
